# 01 - Data access and first look
Search Sentinel-2 L2A on the Planetary Computer, load one scene, inspect RGB, NDVI and FAI. **Verify the pilot bounding box** in `config/config.yaml` covers the bay you want before going further.

On Colab: `!pip install -r requirements.txt` first, then restart the runtime.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from hyacinthwatch.config import load_config
cfg = load_config("../config/config.yaml")
bbox = cfg["aoi"]["pilot_bbox"]
BANDS = cfg["bands"]
from hyacinthwatch.stac_data import search_scenes, load_scene
from hyacinthwatch.indices import bands_dict, compute_indices

In [ ]:
items = search_scenes(bbox, cfg["stac"]["date_range"], cfg["stac"]["max_cloud"])
print(len(items), "scenes;", items[0].datetime.date(), "->", items[-1].datetime.date())

In [ ]:
item = items[len(items) // 2]
stack, scl = load_scene(item, bbox, BANDS)
img = stack.values
idx = compute_indices(bands_dict(img, BANDS))
rgb = np.stack([img[BANDS.index("B04")], img[BANDS.index("B03")], img[BANDS.index("B02")]], -1)
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(np.clip(rgb / 0.3, 0, 1)); ax[0].set_title(str(item.datetime.date()) + " RGB")
ax[1].imshow(idx["ndvi"], cmap="RdYlGn", vmin=-0.2, vmax=0.8); ax[1].set_title("NDVI")
ax[2].imshow(idx["fai"], cmap="viridis", vmin=-0.02, vmax=0.1); ax[2].set_title("FAI")
for a in ax: a.axis("off")
plt.show()

**Check before moving on:** is the lake where you expect? Are there obvious mats (high NDVI and FAI over water)? Note that cyanobacterial blooms can also show high FAI; that confusion is the main risk of the whole project.